# Config A — Dynamic Programming
## Optimal Policy via Policy Iteration & Value Iteration

**Master in Data Science & Advanced Analytics — Reinforcement Learning**

---

This notebook computes the **optimal policy** for the ICU-Sepsis-v2 discrete MDP (716 states × 25 actions) using Dynamic Programming methods:

1. **Policy Iteration** — iterative policy evaluation + policy improvement until convergence
2. **Value Iteration** — direct Bellman optimality backup until convergence

These DP solutions serve as an **upper-bound benchmark** for the model-free methods (Q-Learning, SARSA), since they have access to the full MDP model (transition matrix $P$ and reward matrix $R$).

### Key Questions
- What is the best achievable survival rate and mean return under our configuration?
- How does the optimal policy distribute treatment actions across states?
- Do Policy Iteration and Value Iteration converge to the same policy?

In [1]:
"""
Dynamic Programming for Config A (discrete 716-state sepsis MDP).

Policy Iteration and Value Iteration using the full MDP model (P, R)
accessible via the environment. This gives the optimal policy, which
serves as an upper-bound benchmark for Q-Learning and SARSA.

Usage:
    from envs.env_setup import make_sepsis_env
    env = make_sepsis_env()
    raw = env.unwrapped
    P = raw._tx_mat    # transition matrix [n_states, n_actions, n_states]
    R = raw._r_mat     # reward matrix     [n_states, n_actions, n_states]
"""

import numpy as np

In [2]:
# ---------------------------------------------------------------------------
# Helper: compute expected reward R(s,a) from the 3D reward matrix R(s,a,s')
# ---------------------------------------------------------------------------
# Nas aulas, o R já era R(s,a) — um valor por estado-acção.
# Aqui o ambiente guarda R(s,a,s') — a recompensa para cada transição.
# Para ficar igual às aulas, calculamos o valor esperado:
#   R_expected(s,a) = sum_{s'} P(s,a,s') * R(s,a,s')
# ---------------------------------------------------------------------------

def compute_expected_reward(P, R):
    """
    Converte a matriz de recompensa 3D R(s,a,s') para R_expected(s,a).

    Args:
        P: tensor de transições com shape (n_states, n_actions, n_states)
        R: tensor de recompensas com shape (n_states, n_actions, n_states)

    Returns:
        R_expected: matriz com shape (n_states, n_actions)
    """
    # Para cada (s,a), somamos P(s,a,s') * R(s,a,s') sobre todos os s'
    R_expected = np.sum(P * R, axis=2)
    return R_expected

In [ ]:
# ---------------------------------------------------------------------------
# Policy Evaluation 
# ---------------------------------------------------------------------------
# Dado uma política fixa, calcula quanto vale cada estado seguindo essa
# política. Usa a equação de Bellman da expectativa:
#   V(s) = sum_{s'} P(s'|s,a) * [R(s,a) + gamma * V(s')]
# ---------------------------------------------------------------------------

def compute_value_function(policy, P, R_expected, gamma=1.0):
    """
    Calcula a função de valor para uma política fixa (Policy Evaluation).

    Args:
        policy: array com shape (n_states,) — acção a tomar em cada estado
        P: tensor de transições com shape (n_states, n_actions, n_states)
        R_expected: matriz de recompensas com shape (n_states, n_actions)
        gamma: fator de desconto

    Returns:
        value_table: array com shape (n_states,) — valor de cada estado
    """
    # número de iterações máximo
    num_iterations = 10000

    # threshold para convergência (igual às aulas)
    threshold = 1e-6

    n_states = P.shape[0]

    # inicializamos a value table com zeros (igual às aulas)
    value_table = np.zeros(n_states)

    # para cada iteração
    for i in range(num_iterations):

        # guardamos uma cópia da value table anterior
        updated_value_table = np.copy(value_table)

        # para cada estado
        for s in range(n_states):

            # seleccionamos a acção que a política indica para este estado
            a = int(policy[s])

            # calculamos o valor do estado usando a equação de Bellman:
            # V(s) = sum_{s'} P(s'|s,a) * [R(s,a) + gamma * V(s')]
            value_table[s] = sum([
                P[s, a, s_next] * (R_expected[s, a] + gamma * updated_value_table[s_next])
                for s_next in range(n_states)
            ])

        # verificamos se a diferença entre a value table atual e a anterior
        # é menor que o threshold — se sim, convergiu e paramos
        if np.max(np.abs(updated_value_table - value_table)) <= threshold:
            break

    return value_table

In [4]:
# ---------------------------------------------------------------------------
# Policy Improvement 
# ---------------------------------------------------------------------------
# Com a value function calculada, extraímos a melhor política:
# para cada estado, escolhemos a acção com maior Q-value.
#   Q(s,a) = sum_{s'} P(s'|s,a) * [R(s,a) + gamma * V(s')]
#   pi(s)  = argmax_a Q(s,a)
# ---------------------------------------------------------------------------

def extract_policy(value_table, P, R_expected, gamma=1.0):
    """
    Extrai a política óptima a partir da função de valor (Policy Improvement).

    Args:
        value_table: array com shape (n_states,) — função de valor actual
        P: tensor de transições com shape (n_states, n_actions, n_states)
        R_expected: matriz de recompensas com shape (n_states, n_actions)
        gamma: fator de desconto

    Returns:
        policy: array com shape (n_states,) — acção óptima em cada estado
    """
    n_states, n_actions, _ = P.shape

    # inicializamos a política com zeros
    policy = np.zeros(n_states)

    # para cada estado
    for s in range(n_states):

        # calculamos o Q-value para cada acção possível
        Q_values = [
            sum([
                P[s, a, s_next] * (R_expected[s, a] + gamma * value_table[s_next])
                for s_next in range(n_states)
            ])
            for a in range(n_actions)
        ]

        # escolhemos a acção com maior Q-value (passo greedy)
        policy[s] = np.argmax(np.array(Q_values))

    return policy

In [5]:
# ---------------------------------------------------------------------------
# Policy Iteration 
# ---------------------------------------------------------------------------
# Alterna entre Policy Evaluation e Policy Improvement até a política
# não mudar — nesse ponto encontrámos a política óptima.
# ---------------------------------------------------------------------------

def policy_iteration(P, R, gamma=1.0):
    """
    Policy Iteration num MDP tabular.

    Args:
        P: tensor de transições com shape (n_states, n_actions, n_states)
        R: tensor de recompensas com shape (n_states, n_actions, n_states)
        gamma: fator de desconto (1.0 por omissão — sem desconto temporal,
               igual à convenção do ambiente ICU-Sepsis)

    Returns:
        policy: array com shape (n_states,) — política óptima
        value_function: array com shape (n_states,) — função de valor óptima
        convergence_deltas: lista com o delta máximo por iteração (para gráfico)
    """
    n_states = P.shape[0]

    # calculamos o R_expected (s,a) a partir do R(s,a,s') do ambiente
    R_expected = compute_expected_reward(P, R)

    # começamos com uma política inicial aleatória (igual às aulas)
    policy = np.random.randint(0, P.shape[1], n_states)

    # inicializamos a value function com zeros
    value_function = np.zeros(n_states)

    # número máximo de iterações
    num_iterations = 1000

    # lista para guardar os deltas de convergência (para o gráfico do relatório)
    convergence_deltas = []

    # para cada iteração
    for i in range(num_iterations):

        # --- PASSO 1: Policy Evaluation ---
        # calculamos a função de valor para a política actual
        value_function = compute_value_function(policy, P, R_expected, gamma)

        # --- PASSO 2: Policy Improvement ---
        # extraímos uma nova política melhorada
        new_policy = extract_policy(value_function, P, R_expected, gamma)

        # guardamos o delta para o gráfico de convergência
        delta = np.max(np.abs(value_function - np.zeros(n_states)))
        convergence_deltas.append(delta)

        # se a política não mudou, convergiu — paramos
        if np.all(policy == new_policy):
            print(f'Policy Iteration convergiu após {i + 1} iterações!')
            break

        # caso contrário, actualizamos a política
        policy = new_policy

    return policy.astype(int), value_function, convergence_deltas

In [6]:
# ---------------------------------------------------------------------------
# Value Iteration 
# ---------------------------------------------------------------------------
# Em vez de avaliar e melhorar alternadamente, actualiza directamente
# o valor de cada estado tomando o máximo sobre todas as acções:
#   V(s) = max_a sum_{s'} P(s'|s,a) * [R(s,a) + gamma * V(s')]
# No fim, extrai a política de uma só vez.
# ---------------------------------------------------------------------------

def value_iteration(P, R, gamma=1.0):
    """
    Value Iteration num MDP tabular.

    Args:
        P: tensor de transições com shape (n_states, n_actions, n_states)
        R: tensor de recompensas com shape (n_states, n_actions, n_states)
        gamma: fator de desconto

    Returns:
        policy: array com shape (n_states,) — política óptima
        value_function: array com shape (n_states,) — função de valor óptima
        convergence_deltas: lista com o delta máximo por iteração (para gráfico)
    """
    n_states, n_actions, _ = P.shape

    # calculamos o R_expected (s,a) a partir do R(s,a,s') do ambiente
    R_expected = compute_expected_reward(P, R)

    # número máximo de iterações
    num_iterations = 10000

    # threshold para convergência (igual às aulas)
    threshold = 1e-6

    # inicializamos a value function com zeros
    value_function = np.zeros(n_states)

    # lista para guardar os deltas de convergência (para o gráfico do relatório)
    convergence_deltas = []

    # para cada iteração
    for i in range(num_iterations):

        # guardamos uma cópia da value function anterior
        updated_value_function = np.copy(value_function)

        # para cada estado
        for s in range(n_states):

            # calculamos o Q-value para cada acção possível
            Q_values = [
                sum([
                    P[s, a, s_next] * (R_expected[s, a] + gamma * updated_value_function[s_next])
                    for s_next in range(n_states)
                ])
                for a in range(n_actions)
            ]

            # actualizamos V(s) com o máximo dos Q-values (Bellman optimality)
            value_function[s] = np.max(Q_values)

        # calculamos o delta máximo entre iterações
        delta = np.max(np.abs(updated_value_function - value_function))
        convergence_deltas.append(delta)

        # se o delta é menor que o threshold, convergiu — paramos
        if delta <= threshold:
            print(f'Value Iteration convergiu após {i + 1} iterações!')
            break

    # no fim, extraímos a política óptima a partir da value function final
    policy = extract_policy(value_function, P, R_expected, gamma)

    return policy.astype(int), value_function, convergence_deltas


# ---------------------------------------------------------------------------
# get_policy — converte o array da política numa função chamável
# ---------------------------------------------------------------------------
# Necessário para usar com utils.evaluation.eval_agent()

def get_policy(policy_array):
    """
    Transforma o array da política numa função chamável.
    Necessário para a função eval_agent() de utils/evaluation.py.

    Args:
        policy_array: array com shape (n_states,)

    Returns:
        função que recebe um estado (int) e devolve uma acção (int)
    """
    return lambda obs: int(policy_array[obs])

In [8]:
# 1. Criar o ambiente e extrair P e R
from envs.env_setup import make_sepsis_env
env = make_sepsis_env()
raw = env.unwrapped
P = raw._tx_mat
R = raw._r_mat

ModuleNotFoundError: No module named 'envs'

In [ ]:
# 1. Criar o ambiente e extrair P e R
from envs.env_setup import make_sepsis_env
env = make_sepsis_env()
raw = env.unwrapped
P = raw._tx_mat
R = raw._r_mat

# 2. Correr os algoritmos
from agents.config_a.dynamic_programming import policy_iteration, value_iteration, get_policy

policy_pi, V_pi, deltas_pi = policy_iteration(P, R, gamma=1.0)
policy_vi, V_vi, deltas_vi = value_iteration(P, R, gamma=1.0)

# 3. Medir com a função oficial
from utils.evaluation import eval_agent, print_results

results_pi = eval_agent(get_policy(policy_pi), make_sepsis_env, n_eval_episodes=1000, seed=42)
results_vi = eval_agent(get_policy(policy_vi), make_sepsis_env, n_eval_episodes=1000, seed=42)

print_results("Policy Iteration", results_pi)
print_results("Value Iteration",  results_vi)

# 4. Gráficos
from utils.plotting import plot_comparison
import matplotlib.pyplot as plt

# Gráfico de comparação de survival rate
plot_comparison(
    {"Policy Iteration": results_pi, "Value Iteration": results_vi},
    metric="survival_rate",
    title="Config A — DP Algorithms Comparison"
)
plt.show()

# Curvas de convergência
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(deltas_pi)
axes[0].set_title("Policy Iteration — Convergência")
axes[0].set_xlabel("Iteração")
axes[0].set_ylabel("Delta máximo")
axes[0].grid(alpha=0.3)

axes[1].plot(deltas_vi, color="tab:orange")
axes[1].set_title("Value Iteration — Convergência")
axes[1].set_xlabel("Iteração")
axes[1].set_ylabel("Delta máximo")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 0. Setup & Imports

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
import os
import time

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

os.makedirs('plots', exist_ok=True)
os.makedirs('results/config_a', exist_ok=True)
PLOTS_DIR = 'plots'
RESULTS_DIR = 'results/config_a'

SEED = 42
np.random.seed(SEED)

# Import constants and env factory from env_setup.py
from envs.env_setup import (
    ENV_ID, N_STATES, N_ACTIONS, STATE_SURVIVED, STATE_DIED,
    GAMMA, INTENSITY, SOFA_BIAS, LAM,
    make_sepsis_env,
)

# Import DP algorithms
from agents.config_a.dynamic_programming import (
    policy_iteration, value_iteration, get_policy
)

# Import evaluation utilities
from utils.evaluation import eval_agent, print_results
from utils.plotting import plot_training_curve, plot_comparison

print(f'ICU-Sepsis-v2 | States: {N_STATES} | Actions: {N_ACTIONS}')
print(f'Terminal states: {STATE_SURVIVED} (survived, r=+1)  {STATE_DIED} (died, r=0)')
print(f'Configuration: sofa_bias={SOFA_BIAS}, lam={LAM}, gamma={GAMMA}')
print('Setup complete!')

ModuleNotFoundError: No module named 'icu_sepsis'

---
## 1. Extract the MDP Model

Dynamic Programming requires full knowledge of the MDP — the transition matrix $P(s'|s,a)$ and the expected reward $R(s,a)$. The ICU-Sepsis environment exposes these as internal attributes.

In [ ]:
# Create the environment with our configuration
env = make_sepsis_env()

# Extract the full MDP model
raw = env.unwrapped
P = raw._tx_mat                          # shape (716, 25, 716) — P[s,a,s'] = P(s'|s,a)
R_sasp = raw._r_mat                      # (716, 25, 716) — R[s, a, s']
R = (P * R_sasp).sum(axis=2)             # (716, 25) — E[r | s, a]

print(f'Transition matrix P shape: {P.shape}  (S × A × S\')')
print(f'Reward matrix R shape    : {R.shape}  (S × A)')
print(f'Reward range             : [{R.min():.4f}, {R.max():.4f}]')
print()

# Verify transition probabilities sum to 1 for all (s, a) pairs
p_sums = P.sum(axis=2)
print(f'P row sums — min: {p_sums.min():.6f}, max: {p_sums.max():.6f} (should be ~1.0)')

---
## 2. Random Baseline

Before computing optimal policies, we establish the **random baseline** — the performance floor that all algorithms must beat.

In [ ]:
# Random baseline using eval_agent
random_policy = lambda obs: np.random.randint(N_ACTIONS)

print('Evaluating random baseline (1000 episodes)...')
random_results = eval_agent(
    policy_fn=random_policy,
    env_factory=lambda: make_sepsis_env(verbose=False),
    n_eval_episodes=1000,
    seed=SEED
)
print_results('Random Baseline', random_results)
print()
print('All DP solutions must beat these numbers.')

---
## 3. Policy Iteration

Policy Iteration alternates between two steps:
1. **Policy Evaluation**: compute $V^\pi$ for the current policy $\pi$ by solving the Bellman equation iteratively.
2. **Policy Improvement**: update the policy greedily with respect to $V^\pi$.

The algorithm is guaranteed to converge to the optimal policy in a finite number of iterations.

In [ ]:
print('Running Policy Iteration...')
start = time.time()
pi_policy, pi_V = policy_iteration(P, R, gamma=GAMMA, tol=1e-10)
pi_time = time.time() - start

print(f'✓ Policy Iteration converged in {pi_time:.3f} seconds')
print(f'  Value function range: [{pi_V.min():.4f}, {pi_V.max():.4f}]')
print(f'  Mean V(s) across non-terminal states: {pi_V[:N_STATES-2].mean():.4f}')
print(f'  Unique actions used: {len(np.unique(pi_policy[:N_STATES-2]))}/{N_ACTIONS}')

### 3.1 Evaluate Policy Iteration Policy

In [ ]:
print('Evaluating Policy Iteration optimal policy (1000 episodes)...')
pi_results = eval_agent(
    policy_fn=get_policy(pi_policy),
    env_factory=lambda: make_sepsis_env(verbose=False),
    n_eval_episodes=1000,
    seed=SEED
)
print_results('Policy Iteration (Optimal)', pi_results)

---
## 4. Value Iteration

Value Iteration directly applies the Bellman optimality backup:
$$V(s) \leftarrow \max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a) V(s') \right]$$

until $\|V_{k+1} - V_k\|_\infty < \epsilon$. The policy is then extracted greedily from the converged $V$.

In [ ]:
print('Running Value Iteration...')
start = time.time()
vi_policy, vi_V = value_iteration(P, R, gamma=GAMMA, tol=1e-10)
vi_time = time.time() - start

print(f'✓ Value Iteration converged in {vi_time:.3f} seconds')
print(f'  Value function range: [{vi_V.min():.4f}, {vi_V.max():.4f}]')
print(f'  Mean V(s) across non-terminal states: {vi_V[:N_STATES-2].mean():.4f}')
print(f'  Unique actions used: {len(np.unique(vi_policy[:N_STATES-2]))}/{N_ACTIONS}')

### 4.1 Evaluate Value Iteration Policy

In [ ]:
print('Evaluating Value Iteration optimal policy (1000 episodes)...')
vi_results = eval_agent(
    policy_fn=get_policy(vi_policy),
    env_factory=lambda: make_sepsis_env(verbose=False),
    n_eval_episodes=1000,
    seed=SEED
)
print_results('Value Iteration (Optimal)', vi_results)

---
## 5. Comparison: PI vs VI

Both methods should converge to the same optimal policy. Let's verify this and compare their performance.

In [ ]:
# Compare policies
policies_match = np.all(pi_policy == vi_policy)
n_diff = np.sum(pi_policy != vi_policy)

print('=== Policy Iteration vs Value Iteration ===')
print(f'  Policies identical: {policies_match}')
if not policies_match:
    print(f'  States with different actions: {n_diff}/{N_STATES}')
print()

# Compare value functions
v_diff = np.max(np.abs(pi_V - vi_V))
print(f'  Max |V_PI - V_VI|: {v_diff:.2e}')
print(f'  Mean |V_PI - V_VI|: {np.mean(np.abs(pi_V - vi_V)):.2e}')
print()

# Computation time comparison
print(f'  Policy Iteration time: {pi_time:.3f}s')
print(f'  Value Iteration time:  {vi_time:.3f}s')
print(f'  Speed ratio (PI/VI):   {pi_time/vi_time:.2f}x')

---
## 6. Analysis of the Optimal Policy

### 6.1 Action Distribution

Analyse how the optimal policy distributes treatment actions across states. Each action corresponds to a combination of vasopressor level (0–4) and IV fluid level (0–4):
$$\text{action} = \text{vaso\_level} \times 5 + \text{fluid\_level}$$

In [ ]:
# Use Policy Iteration policy as the reference optimal policy
optimal_policy = pi_policy
optimal_V = pi_V

# Analyse action distribution (non-terminal states only)
clinical_actions = optimal_policy[:N_STATES - 2]
action_counts = np.bincount(clinical_actions, minlength=N_ACTIONS)

# Decode actions into vasopressor and fluid levels
vaso_levels = clinical_actions // 5
fluid_levels = clinical_actions % 5

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Action frequency bar chart
ax = axes[0]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, N_ACTIONS))
ax.bar(range(N_ACTIONS), action_counts, color=colors)
ax.set_xlabel('Action Index')
ax.set_ylabel('Number of States')
ax.set_title('Optimal Policy — Action Frequency')
ax.grid(axis='y', alpha=0.3)

# Vasopressor level distribution
ax = axes[1]
vaso_counts = np.bincount(vaso_levels, minlength=5)
ax.bar(range(5), vaso_counts, color='steelblue', alpha=0.8)
ax.set_xlabel('Vasopressor Level')
ax.set_ylabel('Number of States')
ax.set_title('Vasopressor Level Distribution')
ax.set_xticks(range(5))
ax.grid(axis='y', alpha=0.3)

# IV fluid level distribution
ax = axes[2]
fluid_counts = np.bincount(fluid_levels, minlength=5)
ax.bar(range(5), fluid_counts, color='coral', alpha=0.8)
ax.set_xlabel('IV Fluid Level')
ax.set_ylabel('Number of States')
ax.set_title('IV Fluid Level Distribution')
ax.set_xticks(range(5))
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Optimal Policy — Treatment Distribution (Dynamic Programming)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/dp_action_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Most common action: {np.argmax(action_counts)} '
      f'(vaso={np.argmax(action_counts)//5}, fluid={np.argmax(action_counts)%5}) '
      f'— used in {action_counts.max()} states')

### 6.2 Treatment Heatmap

Visualise the joint distribution of vasopressor and IV fluid levels in the optimal policy as a 2D heatmap.

In [ ]:
# Create 2D heatmap of treatment combinations
treatment_matrix = np.zeros((5, 5), dtype=int)
for a in clinical_actions:
    v, f = a // 5, a % 5
    treatment_matrix[v, f] += 1

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    treatment_matrix, annot=True, fmt='d', cmap='YlOrRd',
    xticklabels=[f'Fluid {i}' for i in range(5)],
    yticklabels=[f'Vaso {i}' for i in range(5)],
    ax=ax, cbar_kws={'label': 'Number of States'}
)
ax.set_title('Optimal Policy — Treatment Combinations Heatmap', fontsize=13, fontweight='bold')
ax.set_xlabel('IV Fluid Level')
ax.set_ylabel('Vasopressor Level')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/dp_treatment_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Value Function Analysis

Examine the distribution of state values under the optimal policy. States with higher values correspond to clinical situations where survival is more likely.

In [ ]:
# Value function distribution (non-terminal states)
clinical_V = optimal_V[:N_STATES - 2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of V(s)
ax = axes[0]
ax.hist(clinical_V, bins=50, color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.axvline(clinical_V.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean = {clinical_V.mean():.4f}')
ax.set_xlabel('V(s)')
ax.set_ylabel('Number of States')
ax.set_title('Distribution of Optimal State Values')
ax.legend()
ax.grid(alpha=0.3)

# Sorted value function
ax = axes[1]
sorted_V = np.sort(clinical_V)
ax.plot(sorted_V, color='steelblue', linewidth=1.5)
ax.fill_between(range(len(sorted_V)), sorted_V, alpha=0.2, color='steelblue')
ax.axhline(clinical_V.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean = {clinical_V.mean():.4f}')
ax.set_xlabel('States (sorted by value)')
ax.set_ylabel('V(s)')
ax.set_title('Sorted Optimal Value Function')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Optimal Value Function (Dynamic Programming)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/dp_value_function.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Value function statistics (non-terminal states):')
print(f'  Min V(s):    {clinical_V.min():.4f}')
print(f'  Max V(s):    {clinical_V.max():.4f}')
print(f'  Mean V(s):   {clinical_V.mean():.4f}')
print(f'  Median V(s): {np.median(clinical_V):.4f}')
print(f'  Std V(s):    {clinical_V.std():.4f}')

### 6.4 Treatment Intensity Analysis

Examine the relationship between the optimal treatment intensity and the state values. The treatment intensity is defined as:
$$\text{intensity}(a) = \frac{\text{vaso\_level} + \text{fluid\_level}}{8}$$

The `lam` penalty in the reward function encourages parsimonious treatment (lower intensity), so we expect the optimal policy to favour lower-intensity treatments unless higher intensity significantly improves survival.

In [ ]:
# Compute treatment intensity for the optimal policy
optimal_intensity = INTENSITY[clinical_actions]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: V(s) vs Treatment Intensity
ax = axes[0]
ax.scatter(clinical_V, optimal_intensity, alpha=0.3, s=10, color='steelblue')
ax.set_xlabel('V(s) — State Value')
ax.set_ylabel('Treatment Intensity')
ax.set_title('State Value vs Treatment Intensity')
ax.grid(alpha=0.3)

# Distribution of treatment intensity
ax = axes[1]
unique_intensities, intensity_counts = np.unique(optimal_intensity, return_counts=True)
ax.bar(unique_intensities, intensity_counts, width=0.05, color='coral', alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Treatment Intensity')
ax.set_ylabel('Number of States')
ax.set_title('Distribution of Treatment Intensity in Optimal Policy')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Treatment Intensity Analysis (Dynamic Programming)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/dp_treatment_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Treatment intensity statistics:')
print(f'  Mean intensity:   {optimal_intensity.mean():.4f}')
print(f'  Median intensity: {np.median(optimal_intensity):.4f}')
print(f'  Min intensity:    {optimal_intensity.min():.4f}')
print(f'  Max intensity:    {optimal_intensity.max():.4f}')

---
## 7. Sensitivity Analysis: Effect of Discount Factor (γ)

Although the ICU-Sepsis convention uses $\gamma = 1.0$, it's instructive to see how the optimal policy changes with different discount factors.

In [ ]:
gammas = [0.90, 0.95, 0.99, 1.0]
gamma_results = {}

for g in gammas:
    print(f'\nγ = {g}:')
    policy_g, V_g = value_iteration(P, R, gamma=g, tol=1e-10)
    
    results_g = eval_agent(
        policy_fn=get_policy(policy_g),
        env_factory=lambda: make_sepsis_env(verbose=False),
        n_eval_episodes=1000,
        seed=SEED
    )
    gamma_results[f'γ={g}'] = results_g
    print_results(f'  VI (γ={g})', results_g)
    
    # Policy similarity to γ=1.0
    if g != 1.0:
        match_frac = np.mean(policy_g == vi_policy)
        print(f'  Policy match with γ=1.0: {match_frac:.1%}')

In [ ]:
# Plot gamma sensitivity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean return by gamma
ax = axes[0]
g_labels = list(gamma_results.keys())
g_returns = [gamma_results[k]['mean_return'] for k in g_labels]
g_stds = [gamma_results[k]['std_return'] for k in g_labels]
bars = ax.bar(g_labels, g_returns, color='steelblue', alpha=0.8, yerr=g_stds, capsize=5)
ax.bar_label(bars, fmt='%.4f', padding=5)
ax.set_ylabel('Mean Return')
ax.set_title('Mean Return vs Discount Factor')
ax.grid(axis='y', alpha=0.3)

# Survival rate by gamma
ax = axes[1]
g_survival = [gamma_results[k]['survival_rate'] * 100 for k in g_labels]
bars = ax.bar(g_labels, g_survival, color='coral', alpha=0.8)
ax.bar_label(bars, fmt='%.1f%%', padding=5)
ax.set_ylabel('Survival Rate (%)')
ax.set_title('Survival Rate vs Discount Factor')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Sensitivity to Discount Factor γ', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/dp_gamma_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Save Optimal Policy & Results

Save the optimal policy, value function, and evaluation results for later use when comparing against Q-Learning and SARSA.

In [ ]:
# Save optimal policy and value function
np.save(f'{RESULTS_DIR}/dp_optimal_policy.npy', pi_policy)
np.save(f'{RESULTS_DIR}/dp_optimal_V.npy', pi_V)

print(f'✓ Optimal policy saved to {RESULTS_DIR}/dp_optimal_policy.npy')
print(f'✓ Optimal value function saved to {RESULTS_DIR}/dp_optimal_V.npy')
print()

# Save results summary
results_summary = pd.DataFrame({
    'Algorithm': ['Random Baseline', 'Policy Iteration', 'Value Iteration'],
    'Mean Return': [random_results['mean_return'], pi_results['mean_return'], vi_results['mean_return']],
    'Std Return': [random_results['std_return'], pi_results['std_return'], vi_results['std_return']],
    'Survival Rate': [f"{random_results['survival_rate']:.1%}", f"{pi_results['survival_rate']:.1%}", f"{vi_results['survival_rate']:.1%}"],
    'Mean Ep Length': [random_results['mean_ep_length'], pi_results['mean_ep_length'], vi_results['mean_ep_length']],
    'Compute Time (s)': ['-', f'{pi_time:.3f}', f'{vi_time:.3f}']
})
results_summary.to_csv(f'{RESULTS_DIR}/dp_results_summary.csv', index=False)

print('Results Summary:')
print(results_summary.to_string(index=False))

---
## 9. Summary & Comparison Bar Chart

Final summary comparing the DP optimal policies against the random baseline.

In [ ]:
# Final comparison bar chart
all_results = {
    'Random': random_results,
    'Policy Iteration': pi_results,
    'Value Iteration': vi_results
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean Return comparison
ax = axes[0]
labels = list(all_results.keys())
returns = [all_results[k]['mean_return'] for k in labels]
bar_colors = ['#95a5a6', '#2ecc71', '#27ae60']
bars = ax.bar(labels, returns, color=bar_colors, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.bar_label(bars, fmt='%.4f', padding=5)
ax.set_ylabel('Mean Return')
ax.set_title('Mean Return — DP vs Random Baseline')
ax.grid(axis='y', alpha=0.3)

# Survival Rate comparison
ax = axes[1]
survivals = [all_results[k]['survival_rate'] * 100 for k in labels]
bars = ax.bar(labels, survivals, color=bar_colors, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.bar_label(bars, fmt='%.1f%%', padding=5)
ax.set_ylabel('Survival Rate (%)')
ax.set_title('Survival Rate — DP vs Random Baseline')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Dynamic Programming Optimal Policy vs Random Baseline', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/dp_vs_random_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Key Takeaways ===')
improvement = (pi_results['mean_return'] - random_results['mean_return']) / random_results['mean_return'] * 100
print(f'  Mean return improvement over random: {improvement:+.1f}%')
print(f'  Survival rate: {random_results["survival_rate"]:.1%} (random) → {pi_results["survival_rate"]:.1%} (optimal)')
print(f'  Policy Iteration ≡ Value Iteration: {policies_match}')
print(f'\n  The DP optimal policy provides the upper-bound benchmark for Q-Learning and SARSA.')